Kraus operators represent a discrete, step-wise quantum channel, mapping an initial state to a final state after an interaction whereas The Lindblad master equation describes a smooth, continuous evolution of the density matrix over time.

linblad master equation is given by- d(rho)/dt = -i/ħ[H, rho] + ∑ɣ(L_k(rho)L_k_dagger -1/2{L_k_dagger*L_k, rho})


This equation can be breakdown into 3 parts
1. -i/ħ[H, rho] --> Describes the coherent, unitary evolution driven by the system's Hamiltonian \(H\) (reversible quantum rotations).

2. ɣ(L_k(rho)L_k_dagger) --> Describes discrete "quantum jumps" or stochastic noise events caused by the environment

3. -1/2ɣ{L_k_dagger*L_k, rho} --> Describes continuous, non-unitary population decay. It tracks the steady loss of probability from a state while time passes without a quantum jump occurring.

You need to install qutip in colab environment as it is not pre installed

In [ ]:
%pip install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 70.4 MB/s eta 0:00:00


now import the libraries

In [ ]:
import numpy as np
import qutip as qt

In [ ]:
def validate_T1_T2(T1,T2):
  if T2> 2*T1 :
    raise ValueError(f"for parameters T1, T2 T2={T2} <=2*T1 = {2*T1} must be true")

this is used to avoid invalid cases

In [ ]:
def relaxation_rate(T1):
  #gamma1 = 1/T1 (energy relaxation rate).
  if T1 <= 0:
    raise ValueError(f"T1 must be positive, got {T1}")
  return 1.0/T1

In [ ]:
def dephasing_rate(T1,T2):
  """pure dephasing rate gamma_phi solving:
  1/T2 = 1/(2*T1) + 1/T_phi => gamma_phi = 1/T2 -1/(2*T1)"""
  validate_T1_T2(T1, T2)
  gamma_phi = 1.0/T2 - 1.0/(2*T1)
  return gamma_phi

Now let us build T1 energy relaxation Lindblad operator:
      L_relax = sqrt(gamma1) *sigma_minus where gamma1 = 1/T1
      and sigma_minus = |0><1| lowers the qubit from |1> to |0>, modeling sponataneous energy decay.

In [ ]:
def relaxation_operator(T1):
  """parameters
  T1: energy relaxation time
  returns
  relaxation collapse operator"""
  gamma1 = relaxation_rate(T1)
  return np.sqrt(gamma1)*qt.sigmap()
  #here in traditional quantum mechanics textbooks basis is ordered by spin along z-axis so sigmam() refers to [[0,0], [1,0]]
  #but in qutip this state [[0,0],[1,0]] is represented by sigmap()

Now let us build pure-dephasing lindblad operator:

L_dephase = sqrt(gamma_phi/2) * sigma_z , where gamma_phi = 1/T2 - 1/(2*T1)    


"sigma_z" dephases the qubit (randomizes phase) without changing populations, modeling only T2(non-T1) decoherence sources.

In [ ]:
def dephasing_operator(T1, T2):
  """Parameters:
  T1, T2
  returns: The dephasing collapse operator"""
  validate_T1_T2(T1, T2)
  gamma_phi = dephasing_rate(T1, T2)
  return np.sqrt(gamma_phi/2)*qt.sigmaz()

Now let us build the hamiltonian operator

In [ ]:
def build_hamiltonian(detuning=0.0, dim=2):
  """Parameters
  detuning: angular frequency detuning between qubit and rotating frame. Default 0.0 gives the zero Hamiltonian (pure decoherence, no coherent dynamics)
  dim: hilbert space dimension. only dim=2 (qubit) is currently supported; kept as a parameter for future qubit extension
  returns:
  hamiltonian operator"""
  if detuning == 0.0:
    return 0*qt.qeye(dim)
  return 0.5*detuning*qt.sigmaz()

by composing relaxation_operator() and dephasing_operator() for combined decay of T1/T2 let us build lindblad collapse operator {L_k}

In [ ]:
def collapse_operators(T1, T2):
  """Parameters
  T1: Energy relaxation time
  T2: Total dephasing time
  returns:
  list of all collapse operators
  [L_relax, L_dephase]"""
  validate_T1_T2(T1, T2)
  c_ops = [relaxation_operator(T1)]
  L_dephase = dephasing_operator(T1, T2)
  if L_dephase != 0:
    c_ops.append(L_dephase)
  return c_ops

In [ ]:
def check_lindblad_validity(c_ops):
  if len(c_ops) == 0:
    raise ValueError("Lindblad collapse operators list is empty")
  dim = c_ops[0].shape[0]
  for L in c_ops:
    if not isinstance(L, qt.Qobj):
      raise ValueError("Lindblad collapse operator must be qutip.Qobj instances")
      #to jusr check L is in Qobj form or normal matrix form
    if L.shape[0] != dim or L.shape[1] != dim:
       raise ValueError(f"Lindblad collapse operator must be a square matrix of dimension {dim}")
    if not np.all(np.isfinite(L.full())) :
      raise ValueError("Lindblad collapse operator must be finite")
    return True

now let us evolve lindblad master equation using qutip

In [ ]:
def evolve_lindblad(rho0, times, c_ops, H=None, e_ops=None):
  """Parameters:
  rho0 : initial density matrix
  times: Time points at which to evaluate the solution
  c_ops: collapse (lindblad) operators
  H    : System hamiltonian. Defaults to the zero Hamiltonian (pure decoherence, no coherent dynamics)
  e_ops: expectation-value operators to track. If None, full density matrices are returned at each time point.

  Returns:
  result"""
  check_lindblad_validity(c_ops)
  dim = rho0.shape[0]
  if H is None:
    H = 0*qt.qeye(dim)
    #if user haven't provide any hamiltonian creates a null matrix of size dim
    #0*identiy because no function in qutip to create a null matrix
  result = qt.mesolve(H, rho0, times, c_ops=c_ops, e_ops=e_ops or [] )
  """numerically integrate the lindblad master equation from the initial density matrix
  rho0 over the specified time grid using the given hamiltonian and collapse operators"""
  return result

qutip.mesolve() - is used for both evolution of schrodinger equation(if no collapse operators are given) , lindblad master equation(if collapse operators were given)

it automatically determines which equation to use

now let us simulate the qubit through combined T1/T2 decay

In [ ]:
def simulate_relaxation_lindblad(T1, T2, t_max, n_points=50, detuning=0.0):
  """Parameters
  T1 : energy relaxation time
  T2 : dephasing time
  t_max : final simulation time
  n_points : number of time points
  detuning : (optional) Angular frequency detuning (adds H = 0.5*detuning*sigma_z)
  Returns
  times: array of simulation time points
  pop1 : excited state|1> population
  coherence : array of T2 coherence decay curve"""
  validate_T1_T2(T1, T2)
  times = np.linspace(0, t_max, n_points)
  #divides time between 0 to t_max to into n_points equal parts
  c_ops = collapse_operators(T1, T2)
  H = build_hamiltonian(detuning=detuning)
  #if detuning !=0 there will be ramsey oscillations
  #T1 curve : start in |1>
  rho1_0 = qt.ket2dm(qt.basis(2,1))
  #gives a density matrix |1><1|(initial state density matrix)
  res1 = evolve_lindblad(rho1_0, times, c_ops, H=H)
  #returns all density matrices
  pop1 = np.array([qt.expect(qt.ket2dm(qt.basis(2,1)), s) for s in res1.states])
  """qt.except(A, B) calculates trace of AB
  and trace of density matrix gives population in excited state |1>"""
  #T2 curve: start in |+>
  plus = (qt.basis(2, 0)+ qt.basis(2,1)).unit()
  #gives |+> by taking (|0>+|1>)/sqrt(2)
  rho2_0 = qt.ket2dm(plus)
  #|+><+|
  #off diagonal elements represent coherence
  res2 = evolve_lindblad(rho2_0, times, c_ops, H=H)
  #returns all density matrices
  coherence = np.array([np.abs(s.full()[0, 1]) for s in res2.states])
  """s.full() --> converts qutip object into numpy matrix
  np.abs() gives magnitude of complex number"""
  return times, pop1, coherence


use linear regression and

Fit T1 from population decay: P1(t) = exp(-t/T1)


Fit T2 from coherence decay : |rho01(t)| = 0.5*exp(-t/T2)


In [ ]:
def fit_T1_T2_lindblad(times, pop1, coherence):
  """uses linear regression on the log of the data"""
  times = np.asarray(times)
  pop1 = np.asarray(pop1)
  coherence = np.asarray(coherence)
  mask1 = pop1> 1e-10
  if mask1.sum() < 2:
    raise ValueError("Not enough data points to fit T1")
  slope1, const = np.polyfit(times[mask1], np.log(pop1[mask1]), 1)
  if slope1 >=0:
   raise ValueError("Population is not decaying check T1/T2 inputs")
  T1_fit = -1.0/slope1

  mask2 = coherence > 1e-10
  if mask2.sum() < 2:
   raise ValueError("Not enough data points to fit T2")
  slope2, const = np.polyfit(times[mask2], np.log(coherence[mask2]), 1)
  if slope2 >= 0:
   raise ValueError("Coherence is not decaying check T1/T2 inputs")
  T2_fit = -1.0/slope2
  return T1_fit, T2_fit